In [125]:
from attr import asdict, has
import inspect


class AutoSetterMixin:
    __init_priority__ = ()

    def __attrs_post_init__(self):
        self._apply_private_setters()

    def _apply_private_setters(self):
        fields = []

        for attr_field in self.__attrs_attrs__:
            private_name = attr_field.name

            if not private_name.startswith("_"):
                continue

            public_name = private_name[1:]
            prop = getattr(type(self), public_name, None)

            if isinstance(prop, property) and prop.fset is not None:
                fields.append(public_name)

        priority = getattr(self, "__init_priority__", ())

        ordered_fields = [
            name for name in fields
            if name not in priority
        ]

        ordered_fields += [
            name for name in priority
            if name in fields
        ]

        for public_name in ordered_fields:
            private_name = "_" + public_name
            setattr(self, public_name, getattr(self, private_name))

    def to_dict(self):
        data = {}

        for attr_field in self.__attrs_attrs__:
            name = attr_field.name
            value = getattr(self, name)

            if isinstance(value, AutoSetterMixin):
                data[name] = value.to_dict()
            elif has(value.__class__):
                data[name] = asdict(value)
            else:
                data[name] = value

        properties = {
            name: getattr(self, name)
            for name, value in inspect.getmembers(type(self))
            if isinstance(value, property)
        }

        data.update(properties)

        return data

In [131]:
from attr import define


@define
class RevenusData(AutoSetterMixin):
    nb_ch: int = 129
    nb_gu_ch: float = 1.7
    to: float = 0.8
    m_lin: float = 6

    _f_b_ponderation: float = 0.4
    _not_f_b_ponderation: float = 0.6

    _f_b_ca_ht = 533
    _not_f_b_ca_ht = 187

    _f_b_ca_ttc = 587
    _not_f_b_ca_ttc = 224

    f_b_marge: float = 2.6
    not_f_b_marge: float = 1.45

    _nb_ventes_mensuelles: int = 231
    _cl_acheteurs_mois: int = 0.0432


    @property
    def f_b_ca_ht(self):
        return self._f_b_ca_ht


    @f_b_ca_ht.setter
    def f_b_ca_ht(self, value):
        self._f_b_ca_ht = value


    @property
    def not_f_b_ca_ht(self):
        return self._not_f_b_ca_ht


    @not_f_b_ca_ht.setter
    def not_f_b_ca_ht(self, value):
        self._not_f_b_ca_ht = value


    @property
    def f_b_ca_ttc(self):
        return self._f_b_ca_ttc


    @f_b_ca_ttc.setter
    def f_b_ca_ttc(self, value):
        self._f_b_ca_ttc = value


    @property
    def not_f_b_ca_ttc(self):
        return self._not_f_b_ca_ttc


    @not_f_b_ca_ttc.setter
    def not_f_b_ca_ttc(self, value):
        self._not_f_b_ca_ttc = value



    @property
    def f_b_ponderation(self):
        return self._f_b_ponderation

    @f_b_ponderation.setter
    def f_b_ponderation(self, value):
        if (0 <= value <= 1):
            self._f_b_ponderation = value
            self._not_f_b_ponderation = 1 - value

    @property
    def not_f_b_ponderation(self):
        return self._not_f_b_ponderation

    @not_f_b_ponderation.setter
    def not_f_b_ponderation(self, value):
        if (0 <= value <= 1):
            self._not_f_b_ponderation = value
            self._f_b_ponderation = 1 - value

    @property
    def marge_ponderee(self):
        return (
            self.f_b_ponderation * self.f_b_marge
            + self.not_f_b_ponderation * self.not_f_b_marge
        )

    @property
    def ch_occ(self):
        return self.nb_ch * self.to

    @property
    def cl_heb_jour(self):
        return self.nb_ch * self.nb_gu_ch * self.to

    @property
    def cl_heb_mois(self):
        return self.cl_heb_jour * 30.5

    @property
    def cl_acheteurs_mois(self):
        return self._cl_acheteurs_mois
    

    @cl_acheteurs_mois.setter
    def cl_acheteurs_mois(self, value):
        self._cl_acheteurs_mois = value
        self._nb_ventes_mensuelles = self._cl_acheteurs_mois * self.cl_heb_mois

    @property
    def nb_ventes_mensuelles(self):
        return self._nb_ventes_mensuelles
    

    @nb_ventes_mensuelles.setter
    def nb_ventes_mensuelles(self, value):
        self._nb_ventes_mensuelles = value
        self._cl_acheteurs_mois = self._nb_ventes_mensuelles / self.cl_heb_mois


    @property
    def f_b_ca_ht_10pct(self):
        return (self.f_b_ca_ht * 0.1) / self.f_b_ponderation
    

    @property
    def f_b_ca_ttc_10pct(self):
        return (self.f_b_ca_ttc * 0.1) / self.f_b_ponderation
    


    @property
    def not_f_b_ca_ht_10pct(self):
        return (self.not_f_b_ca_ht * 0.1) / self.not_f_b_ponderation
    

    @property
    def not_f_b_ca_ttc_10pct(self):
        return (self.not_f_b_ca_ttc * 0.1) / self.not_f_b_ponderation

In [ ]:
from attr import define


@define
class SimRevenusData(RevenusData):
    _pilote_revenus_data : RevenusData = None
    __init_priority__ = ("pilote_revenus_data",)
    
    @property
    def pilote_revenus_data(self):
        return self._pilote_revenus_data
    

    @property
    def f_b_ca_ht_rule1(self):
        return (self.pilote_revenus_data.f_b_ca_ht /self.pilote_revenus_data.nb_ventes_mensuelles) * self.nb_ventes_mensuelles
    

    @property
    def not_f_b_ca_ht_rule1(self):
        return (self.pilote_revenus_data.not_f_b_ca_ht /self.pilote_revenus_data.nb_ventes_mensuelles) * self.nb_ventes_mensuelles
    

    @property
    def f_b_ca_ttc_rule1(self):
        return (self.pilote_revenus_data.f_b_ca_ttc /self.pilote_revenus_data.nb_ventes_mensuelles) * self.nb_ventes_mensuelles
    

    @property
    def not_f_b_ca_ttc_rule1(self):
        return (self.pilote_revenus_data.not_f_b_ca_ttc /self.pilote_revenus_data.nb_ventes_mensuelles) * self.nb_ventes_mensuelles
    



    @property
    def f_b_ca_ht_rule2(self):
        diff_ponderation = self.f_b_ponderation - self.pilote_revenus_data.f_b_ponderation
        nb_10pct = diff_ponderation / 0.1
        f_b_add = nb_10pct * self.pilote_revenus_data.f_b_ca_ht_10pct
        return self.f_b_ca_ht_rule1 + f_b_add
    

    @property
    def not_f_b_ca_ht_rule2(self):
        diff_ponderation = self.not_f_b_ponderation - self.pilote_revenus_data.not_f_b_ponderation
        nb_10pct = diff_ponderation / 0.1
        not_f_b_add = nb_10pct * self.pilote_revenus_data.not_f_b_ca_ht_10pct
        return self.not_f_b_ca_ht_rule1 + not_f_b_add
    

    @property
    def f_b_ca_ttc_rule2(self):
        diff_ponderation = self.f_b_ponderation - self.pilote_revenus_data.f_b_ponderation
        nb_10pct = diff_ponderation / 0.1
        f_b_add = nb_10pct * self.pilote_revenus_data.f_b_ca_ttc_10pct
        return self.f_b_ca_ttc_rule1 + f_b_add
    

    @property
    def not_f_b_ca_ttc_rule2(self):
        diff_ponderation = self.not_f_b_ponderation - self.pilote_revenus_data.not_f_b_ponderation
        nb_10pct = diff_ponderation / 0.1
        not_f_b_add = nb_10pct * self.pilote_revenus_data.not_f_b_ca_ttc_10pct
        return self.not_f_b_ca_ttc_rule1 + not_f_b_add
    

    

    @pilote_revenus_data.setter
    def pilote_revenus_data(self, value):
        self._pilote_revenus_data = value
        if(self._pilote_revenus_data is not None):
            self.nb_ventes_mensuelles = self._pilote_revenus_data.cl_acheteurs_mois * self.cl_heb_mois
            self.f_b_ca_ht = (self._pilote_revenus_data.f_b_ca_ht /self._pilote_revenus_data.nb_ventes_mensuelles) * self.nb_ventes_mensuelles
            self.not_f_b_ca_ht = (self._pilote_revenus_data.not_f_b_ca_ht /self._pilote_revenus_data.nb_ventes_mensuelles) * self.nb_ventes_mensuelles
            self.f_b_ca_ttc = (self._pilote_revenus_data.f_b_ca_ttc /self._pilote_revenus_data.nb_ventes_mensuelles) * self.nb_ventes_mensuelles
            self.not_f_b_ca_ttc = (self._pilote_revenus_data.not_f_b_ca_ttc /self._pilote_revenus_data.nb_ventes_mensuelles) * self.nb_ventes_mensuelles


srd = SimRevenusData(
    nb_ch=100, nb_gu_ch=1.5, to=0.8, m_lin=3, f_b_ponderation=0.7, not_f_b_ponderation=0.3,
    pilote_revenus_data = RevenusData()
)

d = srd.to_dict()
d.pop("_pilote_revenus_data")
d

{'nb_ch': 100,
 'nb_gu_ch': 1.5,
 'to': 0.8,
 'm_lin': 3,
 '_f_b_ponderation': 0.7,
 '_not_f_b_ponderation': 0.30000000000000004,
 'f_b_marge': 2.6,
 'not_f_b_marge': 1.45,
 '_nb_ventes_mensuelles': 158.00273597811216,
 '_cl_acheteurs_mois': 0.04317014644210715,
 'ch_occ': 80.0,
 'cl_acheteurs_mois': 0.04317014644210715,
 'cl_heb_jour': 120.0,
 'cl_heb_mois': 3660.0,
 'f_b_ca_ht': 364.5690834473324,
 'f_b_ca_ttc': 401.5047879616963,
 'f_b_ponderation': 0.7,
 'marge_ponderee': 2.255,
 'nb_ventes_mensuelles': 158.00273597811216,
 'not_f_b_ca_ht': 127.90697674418604,
 'not_f_b_ca_ttc': 153.21477428180575,
 'not_f_b_ponderation': 0.30000000000000004,
 'pilote_revenus_data': RevenusData(nb_ch=129, nb_gu_ch=1.7, to=0.8, m_lin=6, _f_b_ponderation=0.4, _not_f_b_ponderation=0.6, f_b_marge=2.6, not_f_b_marge=1.45, _nb_ventes_mensuelles=231.0, _cl_acheteurs_mois=0.04317014644210715)}